<a href="https://colab.research.google.com/github/MahnoorAhmed999/Ai-UNO-/blob/main/24i2540.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:

import random
import copy

class Card:
    def __init__(self, color, value):
        self.color = color  # Red, Blue, Green, Yellow
        self.value = value # 0 to  9 , skip

    def is_skip(self):
      return self.value == 'Skip'

    def matches(self, top):
      return self.color == top.color or self.value == top.value
    def __repr__(self):
      return f"{self.color} {self.value}"

#1st step is to shuffle and distrubute the cards
def generate_deck():
    COLORS = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    for c in COLORS:
        for n in range(10): deck.append(Card(c, n))
        for _ in range(2): deck.append(Card(c, 'Skip'))
    random.shuffle(deck)
    return deck

# BASIC RULES
def get_valid_moves(hand, top):
    v = [c for c in hand if c.matches(top)]
    return v if v else ['Draw']

def apply_move(state, pidx, move):
    s = copy.deepcopy(state)
    hand = s['hands'][pidx]

    if move == 'Draw':
        if s['deck']:
            hand.append(s['deck'].pop())
        s['current'] = (pidx + 1) % 3
    else:
        #  Find the index of the card that matches the moves attributes
        card_to_remove = None
        for i, card in enumerate(hand):
            if card.color == move.color and card.value == move.value:
                card_to_remove = i
                break

        if card_to_remove is not None:
            hand.pop(card_to_remove)
            s['top_card'] = move
            # Rule 3: Skip Card Logic
            s['current'] = (pidx + 2) % 3 if move.is_skip() else (pidx + 1) % 3
        else:
            # Fallback in case of logic error: treat as a draw
            if s['deck']: hand.append(s['deck'].pop())
            s['current'] = (pidx + 1) % 3

    return s

# 3. MANUAL PLAYER
def player_manual(state):
    hand, top = state['hands'][state['current']], state['top_card']
    moves = get_valid_moves(hand, top)
    print(f"Top Card: {top} | Your Hand: {hand}")
    if moves == ['Draw']:
      return 'Draw'
    for i, m in enumerate(moves):
      print(f"[{i}] {m}")

    choice = int(input("Choose card index: "))
    return moves[choice]


In [11]:
# 1. EVALUATION FUNCTION
def evaluate(state, pidx, strategy='defensive'):
    hand = state['hands'][pidx]
    opp_avg = sum(len(state['hands'][i]) for i in range(3) if i != pidx) / 2.0


    w_ai, w_opp, w_skip = (7, 2, 5) if strategy == 'defensive' else (4, 4, 2)


    s_count = sum(1 for c in hand if c.is_skip())
    return 50.0 - (w_ai * len(hand)) + (w_opp * opp_avg) + (w_skip * s_count)

# 2. MINIMAX ALGORITHM
def minimax(state, depth, maximizing, ai_player):
    # Base case depth reached or someone won
    if depth == 0 or any(len(h) == 0 for h in state['hands']):
        return evaluate(state, ai_player, 'defensive'), None

    current = state['current']
    moves = get_valid_moves(state['hands'][current], state['top_card'])

    if maximizing:
        best_sc, best_m = float('-inf'), None
        for m in moves:
            sc, _ = minimax(apply_move(state, current, m), depth-1, False, ai_player)
            if sc > best_sc:
              best_sc, best_m = sc, m
        return best_sc, best_m
    else:
        best_sc = float('inf')
        for m in moves:
            sc, _ = minimax(apply_move(state, current, m), depth-1, True, ai_player)
            if sc < best_sc: best_sc = sc
        return best_sc, None

In [12]:
from collections import Counter

def chance_node(state, depth, ai_player):
    deck = state['deck']
    if not deck: return evaluate(state, ai_player, 'offensive')

    total = len(deck)
    counts = Counter((c.color, c.value) for c in deck)
    expected_score = 0.0

    for (color, value), cnt in counts.items():
        prob = cnt / total
        # Simulate drawing this  card LATER
        temp_state = copy.deepcopy(state)
        temp_state['hands'][ai_player].append(Card(color, value))
        temp_state['current'] = (ai_player + 1) % 3
        score, _ = expectimax(temp_state, depth - 1, ai_player)
        expected_score += prob * score
    return expected_score

def expectimax(state, depth, ai_player):
    if depth == 0 or any(len(h) == 0 for h in state['hands']):
        return evaluate(state, ai_player, 'offensive'), None

    current = state['current']
    moves = get_valid_moves(state['hands'][current], state['top_card'])

    if current == ai_player: # MAX Node
        best_sc, best_m = float('-inf'), None
        for m in moves:
            sc = chance_node(state, depth, ai_player) if m == 'Draw' else expectimax(apply_move(state, current, m), depth-1, ai_player)[0]
            if sc > best_sc: best_sc, best_m = sc, m
        return best_sc, best_m
    else: # Random Opponent Node
        m = random.choice(moves)
        return expectimax(apply_move(state, current, m), depth-1, ai_player)

In [13]:
def run_full_game(mode='simulation'):
    deck = generate_deck()
    hands = [[deck.pop() for _ in range(5)] for _ in range(3)]
    state = {'hands': hands, 'top_card': deck.pop(), 'deck': deck, 'current': 0}

    print("--- Game Started ---")
    while all(len(h) > 0 for h in state['hands']):
        cur = state['current']
        print(f"\nTurn: Player {cur+1}")

        if cur == 0: # P1 - Minimax
            _, move = minimax(state, depth=3, maximizing=True, ai_player=0)
        elif cur == 1: # P2 - Expectimax
            _, move = expectimax(state, depth=3, ai_player=1)
        else: # P3 - User or Sim
            if mode == 'manual': move = player_manual(state)
            else: _, move = minimax(state, depth=3, maximizing=True, ai_player=2)

        print(f"Player {cur+1} plays: {move}")
        state = apply_move(state, cur, move)

    print("\n--- GAME OVER ---")
    for i, h in enumerate(state['hands']):
        if len(h) == 0: print(f"Winner is Player {i+1}!")

# To test it:
run_full_game(mode='simulation')

--- Game Started ---

Turn: Player 1
Player 1 plays: Blue 7

Turn: Player 2
Player 2 plays: Blue 2

Turn: Player 3
Player 3 plays: Blue Skip

Turn: Player 2
Player 2 plays: Red Skip

Turn: Player 1
Player 1 plays: Red 5

Turn: Player 2
Player 2 plays: Red 7

Turn: Player 3
Player 3 plays: Draw

Turn: Player 1
Player 1 plays: Red 2

Turn: Player 2
Player 2 plays: Draw

Turn: Player 3
Player 3 plays: Draw

Turn: Player 1
Player 1 plays: Draw

Turn: Player 2
Player 2 plays: Draw

Turn: Player 3
Player 3 plays: Draw

Turn: Player 1
Player 1 plays: Draw

Turn: Player 2
Player 2 plays: Yellow 2

Turn: Player 3
Player 3 plays: Yellow 1

Turn: Player 1
Player 1 plays: Blue 1

Turn: Player 2
Player 2 plays: Blue 9

Turn: Player 3
Player 3 plays: Green 9

Turn: Player 1
Player 1 plays: Green 6

Turn: Player 2
Player 2 plays: Draw

Turn: Player 3
Player 3 plays: Green 3

Turn: Player 1
Player 1 plays: Draw

Turn: Player 2
Player 2 plays: Draw

Turn: Player 3
Player 3 plays: Green 7

Turn: Player 